# Variational quantum eigensolver for the transverse field Ising model in one dimension

## Imports

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from scipy.optimize import minimize
from IPython.display import clear_output

from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit import Parameter, ParameterVector
from qiskit.circuit.library import hamiltonian_variational_ansatz
from qiskit.primitives import StatevectorEstimator

# Written for Qiskit 2.2.3

## Setting up the Hamiltonian and simulation

The idea of VQE is relatively simple: we want to variationally minimize the energy of an ansatz for the ground state of a Hamiltonian $H$. In classical computing, the size of the state space is a constraint and quantum computing can relieve us of this. For example, an $N$-spin state fits on $N$ qubits but requires $2^N$ digital bits. In VQE, the variational optimization is classical, and only the ansatz preparation and expectation value evaluation are quantum.

The Ising model is a spin model, so it fits natively into a qubit state space*. We will go straight to setting up the Hamiltonian and the simulation. Then we will set up the VQE ansatzes and apply them. Our Hamiltonian is
$$ H = -J \sum_{j=1}^{N-1} Z_j Z_{j+1} - g \sum_{j=1}^N X_j$$
$J$ represents the exchange interaction, which favors aligned neighboring spins, and $g$ represents a magnetic field in the $x$ direction. In the code, we will measure energy in units of $J$ and vary the applied magnetic field $g$ to watch how the ground state evolves.

*Fermion occupation states in compuptational chemistry can be mapped to spin states as well with a Jordan-Wigner transformation. Ironically, the Jordan-Wigner transformation was used in the reverse direction, spin-to-occupation, to calculate the exact ground state energies analytically before the days of quantum computing.

In [ ]:
n_qb = 5
est = StatevectorEstimator()

# Assemble the Hamiltonian from the individual spin operators
def build_Hamiltonian(n_qb):
    ops_ZZ = ["I"*j + "ZZ" + "I"*(n_qb-2-j) for j in range(n_qb-1)]
    coeffs_ZZ = [-1.0]*(n_qb-1)
    ops_X = ["I"*j + "X" + "I"*(n_qb-1-j) for j in range(n_qb)]
    coeffs_X = [-1.0]*n_qb
    return [SparsePauliOp(ops_ZZ, coeffs_ZZ), SparsePauliOp(ops_X, coeffs_X)]

# Run the simulations and get the energies
def calculate_E_and_order(est, ansatz, parameters, H_list, g):
    [H_ZZ, H_X] = H_list
    evs = est.run([(ansatz, [H_ZZ, H_X], parameters)]).result()[0].data.evs
    return [evs[0] + g*evs[1], -evs[1]]

H_list = build_Hamiltonian(n_qb)

## Mean-field analysis and ansatz

There are two limiting cases that give us a good place to start: $g=0$ and $g \gg J$.
- For $g=0$, only the exchange term of the Hamiltonian matters, and it favors aligned spins. Therefore, the ground state $\left|\Psi\right\rangle$ has all spins in the same state: $\left|\Psi\right\rangle = \left|0\right\rangle^{\otimes N}$.
- For $g \gg J$, only the transverse field term of the Hamiltonian matters, and it favors horizontal spins. Therefore, the ground state is $\left|\Psi\right\rangle = \left|+\right\rangle^{\otimes N}$.

At the mean-field level, we take all spins to be in the same state, so $\left|\Psi\right\rangle = \left|\psi\right\rangle^{\otimes N}$ for some single-spin state $\left|\psi\right\rangle$. This means that in the Hamiltonian expectation value, the individual spin operators in the Hamiltonian with an average for each:
$$ \left\langle \Psi_{mf} \right| H \left| \Psi_{mf} \right\rangle = -J(N-1)\left\langle \psi \right| Z \left| \psi \right\rangle^2 - Ng\left\langle \psi \right| X \left| \psi \right\rangle $$
The ground state must balance between the term favoring $Z$ polarization and the term favoring $X$ polarization. The $R_Y$ operator smoothly rotates between $|0\rangle$ and $|+\rangle$, so it's clear that the ground state will have $|\psi\rangle = R_Y(\theta)|0\rangle$ for some $\theta$, so that
$$ \left| \Psi_{mf} \right\rangle = \left( R_Y(\theta)|0\rangle\right)^{\otimes N} $$
Analytically, we can calculate that this is $\theta = \arcsin\left(\frac{Ng}{2(N-1)J}\right)$, but we will instead use VQE to find the ground state at the mean-field level.

While one might think that $\left|\Psi\right\rangle = \left|+\right\rangle^{\otimes N}$ might only be the ground state in the limit $g\to\infty$, the transverse field Ising model has a critical point at which this becomes the ground state for finite $g$. At the mean-field level, using the analytic calculation for $\theta$ above, we reach $\left|\Psi\right\rangle = \left|+\right\rangle^{\otimes N} = \left(R_Y(\theta)|0\rangle\right)^{\otimes N}$ when $\theta = 2\left(1-\frac{1}{N}\right)$. Let's compare this to the results of VQE with this ansatz.

In [ ]:
def build_mean_field_ansatz(n_qb):
    ansatz = QuantumCircuit(n_qb)
    ansatz.ry(Parameter("theta"), range(n_qb))
    return ansatz

def plot_mean_field_results(est, mf_ansatz, H_list):
    g_res = 100
    g = np.linspace(0, 2, g_res)
    theta = np.zeros(g_res)
    E = np.zeros(g_res)
    ord_par = np.zeros(g_res)
    theta_guess = 0
    for j in range(g_res):
        clear_output(wait=True)
        print("Now on {} of {}".format(j+1, g_res))
        E_of_theta_fx = lambda theta: calculate_E_and_order(est, mf_ansatz, theta, H_list, g[j])[0]
        opt_result = minimize(E_of_theta_fx, x0=theta_guess)
        theta[j] = opt_result.x[0]
        E[j] = opt_result.fun
        ord_par[j] = calculate_E_and_order(est, mf_ansatz, theta[j], H_list, g[j])[1]
        # Use this round's minimum for the next guess
        theta_guess = opt_result.x[0]
    clear_output(wait=True)
    fig, axs = plt.subplots(1, 3, figsize=(9,3), layout="constrained")
    axs[0].plot(g, E)
    axs[1].plot(g, theta)
    axs[2].plot(g, ord_par)
    for ax in axs:
        ax.set_xlabel(r"Magnetic field $g$")
    axs[0].set_ylabel("Energy")
    axs[1].set_ylabel(r"$\theta$")
    axs[2].set_ylabel(r"$\langle H_X \rangle$")
    plt.show()

mf_ansatz = build_mean_field_ansatz(n_qb)
plot_mean_field_results(est, mf_ansatz, H_list)

The parameter $\theta$ plays the role not only of the variational parameter, but also of an order parameter which measures the degree to which $|\Psi\rangle$ is favoring spins aligned together in $Z$ or aligned with the $x$-direction magnetic field. It diverges at the critical point, as is standard for order parameters in critical phenomena. $\theta$ is accessible in this case because the ansatz is simple, but for a more intricate ansatz with more parameters we will need a more general indicator of the transition between phases. For this purpose we use $\langle H_X \rangle$ with $H_X = - \sum_{j=1}^N X_j$, which measures the transverse magnetization.

Now let's construct a new ansatz that includes correlations between spins that are neglected at the mean-field level.

## Hamiltonian variational ansatz

The Hamiltonian variational ansatz (HVA) is a standard ansatz that allows for entanglement and correlations. It is constructed by breaking down a Hamiltonian into terms $H_k$ in which each $H_k$ is a sum of commuting Pauli operators. In our case, we have two: $H_{ZZ} = \sum_{j=1}^{N-1} Z_j Z_{j+1}$ and $H_X = - \sum_{j=1}^N X_j$. We have excluded $g$ from $H_X$ here because it can be absorbed into the optimization parameter. The standard HVA is then
$$ \left|\Psi_{hva,0}\right\rangle = \prod_{j=1}^{n_{layers}} \prod_k e^{-i \theta_{k,j} H_k} |0\rangle^{\otimes N} $$
where $n_{layers}$ is a parameter that allows us to increase the ansatz's flexibility - the more layers, the more freedom the ansatz has to vary.

We will modify this slightly and instead apply Hamiltonian evolution to the mean-field ansatz, using
$$ \left|\Psi_{hva}\right\rangle = \prod_{j=1}^{n_{layers}} \left(\prod_k e^{-i \theta_{k,j} H_k}\right) \left(R_Y(\theta)|0\rangle\right)^{\otimes N} $$

In [ ]:
# Construct the Hamiltonian variational ansatz
def build_hamiltonian_variational_ansatz(n_qb, n_layers):
    [H_ZZ, H_X] = build_Hamiltonian(n_qb)
    ansatz = QuantumCircuit(n_qb)
    ansatz.ry(Parameter("theta"), range(n_qb))
    ansatz.compose(hamiltonian_variational_ansatz([H_ZZ, H_X], reps=n_layers), inplace=True)
    return ansatz

def plot_energy_versus_field(est, mf_ansatz, hva_ansatz, H_list, n_layers):
    g_res = 30
    g = np.linspace(0, 2, g_res)
    E_mf = np.zeros(g_res)
    E_hva = np.zeros(g_res)
    E_ed = np.zeros(g_res)
    ord_par_mf = np.zeros(g_res)
    ord_par_hva = np.zeros(g_res)
    ord_par_ed = np.zeros(g_res)
    p_mf_guess = 0
    p_hva_guess = np.zeros(2*n_layers+1)+0.01
    for j in range(g_res):
        clear_output(wait=True)
        print("Now on {} of {}".format(j+1, g_res))
        E_mf_fx = lambda p_mf: calculate_E_and_order(est, mf_ansatz, p_mf, H_list, g[j])[0]
        E_hva_fx = lambda p_hva: calculate_E_and_order(est, hva_ansatz, p_hva, H_list, g[j])[0]
        opt_result_mf = minimize(E_mf_fx, x0=p_mf_guess)
        opt_result_hva = minimize(E_hva_fx, x0=p_hva_guess)
        E_mf[j] = opt_result_mf.fun
        E_hva[j] = opt_result_hva.fun
        ord_par_mf[j] = calculate_E_and_order(est, mf_ansatz, p_mf_guess, H_list, g[j])[1]
        ord_par_hva[j] = calculate_E_and_order(est, hva_ansatz, p_hva_guess, H_list, g[j])[1]
        # Get the exact result from exact diagonalization
        [E_ed_dump, v_ed_dump] = np.linalg.eigh((H_list[0]+g[j]*H_list[1]).to_matrix())
        E_ed[j] = E_ed_dump[0]
        # Ensure that we use the + direction ground state
        ord_par_ed[j] = np.abs(np.matmul(np.matmul( np.reshape(v_ed_dump[:,0].conjugate(), (1,len(v_ed_dump[:,0]))), (H_list[1]).to_matrix()), v_ed_dump[:,0])[0])
        # Use this round's minimum for the next guess
        p_mf_guess = opt_result_mf.x[0]
        p_hva_guess = opt_result_hva.x
    clear_output(wait=True)
    fig, axs = plt.subplots(1, 2, figsize=(6,3), layout="constrained")
    axs[0].plot(g, E_mf, label="Mean field")
    axs[0].plot(g, E_hva, label="HVA")
    axs[0].plot(g, E_ed, label="Exact diag")
    axs[0].set_xlabel(r"Magnetic field $g$")
    axs[0].set_ylabel("Energy")
    axs[0].legend()
    axs[1].plot(g, ord_par_mf, label="Mean field")
    axs[1].plot(g, ord_par_hva, label="HVA")
    axs[1].plot(g, ord_par_ed, label="Exact diag")
    axs[1].set_xlabel(r"Magnetic field $g$")
    axs[1].set_ylabel(r"$\langle H_X \rangle$")
    axs[1].legend()
    plt.show()

n_layers = 2
hva_ansatz = build_hamiltonian_variational_ansatz(n_qb, n_layers)
plot_energy_versus_field(est, mf_ansatz, hva_ansatz, H_list, n_layers)

Since we are using a rather small system for this example, we can also obtain the exact ground state energy by exact diagonalization for comparison. The HVA result even for only a few layers is a substantial improvement over the mean-field ground state energy and closely approximates the exact ground state energy. Furthermore, the mean-field energy is incorrect where the mean-field critical point is. In reality, the transverse-field Ising model's critical point occurs at a lower magnetic field, and it only exists in the limit of an infinite system, $N\to\infty$; for a finite system it is smoothed out. Re-running the simulation with fewer or more sites (fewer or more qubits) will produce a softer or sharper knee in $\langle H_X \rangle$.